# Session 3: a junction simulator, checked against the formula
**A. Real-world problem.** Real cars do not arrive at neat intervals; they come in random
bunches. Does the tidy delay formula of Session 2 still describe them? Does a signal that
reacts to the queue beat a fixed timetable? And what happens when more cars arrive than the
junction can ever serve?

**Plan.** (1) generate seeded random arrivals, (2) simulate the junction second by second
with a fixed plan and a simple actuated rule, (3) validate against the formula and against
conservation of cars, (4) vary the cycle time and the arrival rates, (5) provoke a failure.

*This is a simplified educational model, not a traffic engineering tool.*

In [ ]:
import pathlib
from collections import deque

import numpy as np

OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"
S_SAT = 0.5       # saturation flow, cars per second per lane (a typical textbook value)
AMBER = 3.0       # seconds; no car leaves on amber, so it is the lost time of each phase
LOST = 2 * AMBER  # lost time per cycle for two phases

## Step 1: arrivals
Cars arrive at random with an average rate `q`: the gaps between cars are exponentially
distributed (a Poisson process). A seeded generator makes every run repeatable. For
validation we also keep a perfectly regular stream, which matches the "even arrivals" of the
formula.

In [ ]:
def poisson_arrivals(rng, q, T):
    """Arrival times (s) of a Poisson stream with rate q cars/s over [0, T)."""
    if q <= 0:
        return np.array([])
    n = int(q * T * 1.5 + 50)
    times = np.cumsum(rng.exponential(1.0 / q, size=n))
    while times[-1] < T:
        times = np.concatenate([times, times[-1] + np.cumsum(rng.exponential(1.0 / q, size=n))])
    return times[times < T]


def regular_arrivals(q, T):
    """Evenly spaced arrivals, one every 1/q seconds (the 'no randomness' case)."""
    if q <= 0:
        return np.array([])
    gap = 1.0 / q
    return np.arange(0.5 * gap, T, gap)

## Step 2: two ways to run the signals
Approach `i` belongs to phase `i % 2` (approaches 0 and 2 share phase 0; 1 and 3 share
phase 1). A controller is called every step with the queue lengths and answers
"which phase, and is it green or amber?".
* **Fixed**: a timetable, `green0, amber, green1, amber`, repeated forever.
* **Actuated**: stay green while the phase still has a queue, up to `max_green`; give up the
  green when its queue is empty and the other phase waits, but not before `min_green`.

In [ ]:
class FixedPlan:
    def __init__(self, greens, amber=AMBER):
        self.g = tuple(float(x) for x in greens)
        self.amber = float(amber)
        self.cycle = self.g[0] + self.g[1] + 2 * self.amber

    def __call__(self, t, queues):
        tau = t % self.cycle
        if tau < self.g[0]:
            return 0, True
        if tau < self.g[0] + self.amber:
            return 0, False
        if tau < self.g[0] + self.amber + self.g[1]:
            return 1, True
        return 1, False


class Actuated:
    def __init__(self, min_green=8.0, max_green=45.0, amber=AMBER):
        self.min_green, self.max_green, self.amber = min_green, max_green, amber
        self.phase, self.in_amber, self.timer, self.last_t = 0, False, 0.0, 0.0

    def __call__(self, t, queues):
        dt, self.last_t = t - self.last_t, t
        self.timer += dt
        mine = max([qv for i, qv in enumerate(queues) if i % 2 == self.phase], default=0)
        other = max([qv for i, qv in enumerate(queues) if i % 2 != self.phase], default=0)
        if self.in_amber:
            if self.timer >= self.amber:
                self.phase, self.in_amber, self.timer = 1 - self.phase, False, 0.0
        elif other > 0 and (self.timer >= self.max_green or (self.timer >= self.min_green and mine == 0)):
            self.in_amber, self.timer = True, 0.0
        return self.phase, not self.in_amber

## Step 3: the simulator
Every `dt` seconds: new cars join their queue; the controller picks the phase; cars on a
green approach leave at the saturation flow `s` (a "credit" grows by `s dt` per step and a car
leaves each time it reaches 1; the first car of a green leaves at once). Each car remembers its arrival time, so its delay is exact:
time of leaving minus time of arrival. Cars still waiting at the end count their delay so far.
We record the average delay, the largest queue and the numbers needed to check conservation.

In [ ]:
def simulate(q, controller, s=S_SAT, T=3600.0, dt=0.5, seed=1, poisson=True):
    """Simulate len(q) approaches. q and s may be a list or one number. Returns a dict of results.

    Delay of a car = time it crosses the stop line minus its arrival time (the stop line is
    crossed in the middle of a step, but never before the car arrived).
    """
    q = list(q)
    n = len(q)
    s = [s] * n if np.isscalar(s) else list(s)
    rng = np.random.default_rng(seed)
    arr = [poisson_arrivals(rng, qi, T) if poisson else regular_arrivals(qi, T) for qi in q]
    ptr = [0] * n
    queues = [deque() for _ in range(n)]
    credit = [0.0] * n
    departed, delay_sum = [0] * n, 0.0
    steps = int(round(T / dt))
    qlen = np.zeros((steps, n))
    for k in range(steps):
        t = k * dt
        for i in range(n):
            while ptr[i] < len(arr[i]) and arr[i][ptr[i]] < t + dt:
                queues[i].append(arr[i][ptr[i]])
                ptr[i] += 1
        phase, green = controller(t, [len(x) for x in queues])
        for i in range(n):
            if green and i % 2 == phase:
                credit[i] = min(credit[i] + s[i] * dt, 1.0 + s[i] * dt)
                while credit[i] >= 1.0 and queues[i]:
                    a = queues[i].popleft()
                    delay_sum += max(a, t + dt / 2) - a
                    credit[i] -= 1.0
                    departed[i] += 1
            else:
                credit[i] = 1.0      # the first car of the next green may go at once (no start-up loss)
            qlen[k, i] = len(queues[i])
    waiting = sum(T - a for i in range(n) for a in queues[i])
    total_arrived = sum(len(a) for a in arr)
    return {"arrived": [len(a) for a in arr], "departed": departed, "in_queue": [len(x) for x in queues],
            "avg_delay": (delay_sum + waiting) / max(total_arrived, 1), "max_queue": float(qlen.max()),
            "qlen": qlen, "dt": dt, "total_delay": delay_sum + waiting}

## Step 4: the references to check against
* **Conservation**: every car that arrived has either left or is still in a queue.
* **Uniform delay** (Session 2): `d = C (1 - g/C)^2 / (2 (1 - q/s))` for even arrivals.
* **Webster cycle** (a classic rule of thumb): `C = (1.5 L + 5) / (1 - Y)` where `L` is the lost
  time per cycle and `Y` the sum of the critical flow ratios of the phases; the green is shared
  in proportion to the critical flow ratios of the phases.

In [ ]:
def uniform_delay(C, g, q, s=S_SAT):
    """Seconds per car for even arrivals; infinity if the approach is oversaturated (x > 1)."""
    if g <= 0 or q * C / (g * s) > 1.0 + 1e-12:
        return float("inf")
    return C * (1.0 - g / C) ** 2 / (2.0 * (1.0 - q / s))


def fixed_plan_uniform_delay(q, plan, s=S_SAT):
    """Average of the uniform delays of all approaches, weighted by their flows."""
    s = [s] * len(q) if np.isscalar(s) else list(s)
    num = sum(qi * uniform_delay(plan.cycle, plan.g[i % 2], qi, s[i]) for i, qi in enumerate(q))
    return num / sum(q)


def conservation_gap(result):
    """arrived - departed - still waiting, per approach. Must be zero."""
    return [a - d - w for a, d, w in zip(result["arrived"], result["departed"], result["in_queue"])]


def webster_cycle(Y, lost=LOST):
    """Webster's classic optimum cycle (s); infinity if Y >= 1."""
    return float("inf") if Y >= 1.0 else (1.5 * lost + 5.0) / (1.0 - Y)


def webster_plan(q, s=S_SAT, lost=LOST, amber=AMBER):
    """Fixed two-phase plan from flows: critical flow ratio of each phase = its largest q/s."""
    s = [s] * len(q) if np.isscalar(s) else list(s)
    y = [max(q[i] / s[i] for i in range(len(q)) if i % 2 == ph) for ph in (0, 1)]
    Y = y[0] + y[1]
    C = webster_cycle(Y, lost)
    if not np.isfinite(C):
        raise ValueError("Y >= 1: no fixed plan can serve this demand")
    spare = C - lost
    return FixedPlan((spare * y[0] / Y, spare * y[1] / Y), amber)


def mean_over_seeds(q, make_controller, seeds=range(1, 13), **kw):
    """Average delay and largest queue over several seeds (a fresh controller each time)."""
    runs = [simulate(q, make_controller(), seed=sd, **kw) for sd in seeds]
    return float(np.mean([r["avg_delay"] for r in runs])), float(np.mean([r["max_queue"] for r in runs]))

## Step 5: run the experiments
Two approaches, `q = (0.20, 0.10)` cars/s, `s = 0.5`, one hour per run.

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    q = [0.20, 0.10]
    plan60 = FixedPlan((27.0, 27.0))

    print("--- validation 1: even arrivals against the uniform-delay formula")
    reg = simulate(q, plan60, poisson=False, T=7200.0)
    d_formula = fixed_plan_uniform_delay(q, plan60)
    print(f"simulated {reg['avg_delay']:.2f} s, formula {d_formula:.2f} s, "
          f"difference {100 * (reg['avg_delay'] / d_formula - 1):+.1f} %")
    print("--- validation 2: conservation of cars (arrived - departed - waiting)")
    print("gap per approach:", conservation_gap(reg), "arrived", reg["arrived"])
    print("--- validation 3: delay summed over cars equals the area under the queue curve (Little)")
    area = reg["qlen"].sum() * reg["dt"]
    print(f"sum of delays {reg['total_delay']:.0f} car-s, area under queue {area:.0f} car-s")
    print("--- random arrivals add delay (Poisson, 12 seeds)")
    dr, mqr = mean_over_seeds(q, lambda: plan60)
    print(f"Poisson: {dr:.2f} s average delay, {mqr:.1f} cars largest queue; even arrivals: {reg['avg_delay']:.2f} s, {reg['max_queue']:.0f} cars")

    print("--- sensitivity to cycle length (greens shared 2:1, 12 seeds)")
    Y = sum(qi / S_SAT for qi in q)
    print(f"Webster cycle: Y = {Y:.2f}, C = {webster_cycle(Y):.1f} s")
    cycles = [20, 25, 30, 35, 40, 50, 60, 80, 100, 120]
    sweep = []
    for Cc in cycles:
        spare = Cc - LOST
        res = mean_over_seeds(q, lambda: FixedPlan((spare * 2 / 3, spare / 3)))
        sweep.append(res[0])
        print(f"C = {Cc:3d} s -> average delay {res[0]:5.2f} s, largest queue {res[1]:4.1f} cars")
    print(f"best simulated cycle: {cycles[int(np.argmin(sweep))]} s")

    print("--- fixed (Webster plan) against actuated, demand scaled by f")
    base = webster_plan(q)
    print(f"Webster plan: green {base.g[0]:.1f} / {base.g[1]:.1f} s, C = {base.cycle:.1f} s")
    factors = [0.6, 0.8, 1.0, 1.2, 1.4]
    fixed_d, act_d = [], []
    for f in factors:
        qf = [f * x for x in q]
        fixed_d.append(mean_over_seeds(qf, lambda: base)[0])
        act_d.append(mean_over_seeds(qf, Actuated)[0])
        print(f"f = {f:.1f}: fixed {fixed_d[-1]:6.2f} s, actuated {act_d[-1]:6.2f} s")

    print("--- failure case: demand above capacity (Y > 1)")
    qo = [0.35, 0.20]
    print(f"Y = {sum(x / S_SAT for x in qo):.2f} -> Webster cycle {webster_cycle(sum(x / S_SAT for x in qo))}")
    over = {}
    for name, make in (("fixed 27/27", lambda: plan60), ("actuated", Actuated)):
        r = simulate(qo, make(), T=7200.0)
        over[name] = r
        half = len(r["qlen"]) // 2
        print(f"{name}: waiting cars after 1 h {r['qlen'][half - 1].sum():.0f}, after 2 h {r['qlen'][-1].sum():.0f}")

    fig, ax = plt.subplots(figsize=(6.4, 4))
    ax.plot(cycles, sweep, "o-", label="simulated (Poisson, 12 seeds)")
    ax.axvline(webster_cycle(Y), color="tab:red", ls="--", label="Webster C = 35 s")
    ax.set_xlabel("cycle time C (s)")
    ax.set_ylabel("average delay (s per car)")
    ax.set_title("Delay against cycle length: a shallow valley")
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(OUT / "delay_vs_cycle.png", dpi=120)

    fig2, (a1, a2) = plt.subplots(1, 2, figsize=(9, 3.8))
    a1.plot(factors, fixed_d, "o-", label="fixed (Webster plan)")
    a1.plot(factors, act_d, "s-", label="actuated rule")
    a1.set_xlabel("demand multiplier f (1.0 = 0.20 and 0.10 cars/s)")
    a1.set_ylabel("average delay (s per car)")
    a1.set_title("Fixed against actuated timing")
    a1.grid(alpha=0.3)
    a1.legend(fontsize=8)
    for name, r in over.items():
        a2.plot(np.arange(len(r["qlen"])) * r["dt"] / 60.0, r["qlen"].sum(axis=1), label=name)
    a2.set_xlabel("time (minutes)")
    a2.set_ylabel("cars waiting, both approaches")
    a2.set_title("Failure: demand above capacity (Y = 1.1)")
    a2.grid(alpha=0.3)
    a2.legend(fontsize=8)
    fig2.tight_layout()
    fig2.savefig(OUT / "fixed_vs_actuated.png", dpi=120)
    print("saved figures in", OUT)

## Try it
Do the coding tasks, answer the four questions in the README (section F), then run the
checkpoint. The mini-project at the end of the README asks you to time a junction from counts.

## Coding tasks
1. **Easy.** Run `simulate` with `poisson=True` for seeds 1 to 5 at the plan 27 s / 27 s and print the five delays; explain why they differ.
2. **Stretch.** Change `Actuated(min_green=8.0, max_green=45.0)` to a minimum of 15 s and find out whether the delay at q = (0.20, 0.10) goes up or down.
3. **Explain.** Explain why the actuated rule cannot help when Y is above 1 (use the failure-case figure).

## Assessment
- **A. Conceptual.** Even and Poisson arrivals have the same average flow, yet Poisson arrivals give more delay and longer queues. Why? Could random arrivals ever give less delay than even ones on average?
- **B. Numerical.** A two-phase junction has q = (0.25, 0.15) cars per second, s = 0.5 and L = 6 s. Find Y, Webster's cycle and the two greens. One simulation run reports 720 cars arrived on road 2 and 719 departed. How many are still waiting?
- **C. Graph and simulation interpretation.** In the delay-against-cycle graph (q = 0.20 and 0.10, Webster C = 35 s), describe the curve on the left and on the right of its minimum and say what the simulation finds as the best cycle.
- **D. Design and optimisation challenge.** Design: the junction of the worked example (q = 0.20 and 0.10) was timed with Webster's cycle for its own demand (C = 35 s). In the evening the demand rises to 1.4 times (q = 0.28 and 0.14). The simulation gives 68.5 s average delay with the old plan. Re-time it with Webster's formula, give the new cycle and greens, and say when you would prefer an actuated rule.

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    q = [0.20, 0.10]
    plan60 = L.FixedPlan((27.0, 27.0))

    # --- arrivals
    counts = [len(L.poisson_arrivals(np.random.default_rng(sd), 0.2, 3600.0)) for sd in range(5, 25)]
    checks.append(("Poisson arrivals: the mean over 20 seeds is about q T = 720 cars an hour (within 3 percent)",
                   abs(np.mean(counts) - 720) < 0.03 * 720))
    checks.append(("even arrivals: exactly 720 cars at 0.2 cars/s", len(L.regular_arrivals(0.2, 3600.0)) == 720))
    a1 = L.simulate(q, plan60, seed=3, T=600.0)
    a2 = L.simulate(q, plan60, seed=3, T=600.0)
    a3 = L.simulate(q, plan60, seed=4, T=600.0)
    checks.append(("same seed gives the same result", a1["avg_delay"] == a2["avg_delay"] and a1["arrived"] == a2["arrived"]))
    checks.append(("different seeds give different arrivals", a1["arrived"] != a3["arrived"] or a1["avg_delay"] != a3["avg_delay"]))

    # --- validation 1: even arrivals against the formula (worked example)
    reg = L.simulate(q, plan60, poisson=False, T=7200.0)
    d_formula = L.fixed_plan_uniform_delay(q, plan60)
    checks.append((f"worked example: formula gives 13.86 s (got {d_formula:.2f})", abs(d_formula - 13.8646) < 1e-3))
    checks.append((f"worked example: even-arrival simulation {reg['avg_delay']:.2f} s is within 5 percent of 13.86 s",
                   abs(reg["avg_delay"] / d_formula - 1.0) < 0.05))
    checks.append(("even arrivals: largest queue about q r = 0.2 x 33 = 6.6, so 7 cars", reg["max_queue"] == 7))

    # --- validation 2: conservation, for random arrivals, both controllers, two and four approaches
    ok = True
    for ctrl in (plan60, L.Actuated()):
        r = L.simulate(q, ctrl, seed=2, T=1800.0)
        ok &= L.conservation_gap(r) == [0, 0]
    r4 = L.simulate([0.4, 0.3, 0.35, 0.25], L.Actuated(), s=1.0, seed=2, T=1800.0)
    ok &= L.conservation_gap(r4) == [0, 0, 0, 0]
    checks.append(("conservation: arrived = departed + still waiting (fixed, actuated, four approaches)", bool(ok)))

    # --- validation 3: total delay equals the area under the queue curve
    area = reg["qlen"].sum() * reg["dt"]
    checks.append(("sum of car delays equals area under the queue curve within 3 percent", abs(reg["total_delay"] / area - 1.0) < 0.03))

    # --- randomness
    pois = L.mean_over_seeds(q, lambda: plan60, seeds=range(1, 7), T=3600.0)
    checks.append(("random arrivals give more delay and longer queues than even arrivals",
                   pois[0] > reg["avg_delay"] + 2.0 and pois[1] > reg["max_queue"]))

    # --- edge cases
    z = L.simulate([0.0, 0.0], plan60, T=300.0)
    checks.append(("no cars: zero delay and zero queue, no crash", z["avg_delay"] == 0.0 and z["max_queue"] == 0.0))

    # --- Webster formula and plan
    checks.append(("Webster cycle: (1.5 x 6 + 5) / (1 - 0.6) = 35 s", abs(L.webster_cycle(0.6) - 35.0) < 1e-9))
    wp = L.webster_plan(q)
    checks.append(("Webster plan greens 19.33 s and 9.67 s", abs(wp.g[0] - 58.0 / 3) < 1e-9 and abs(wp.g[1] - 29.0 / 3) < 1e-9))
    try:
        L.webster_plan([0.35, 0.20])
        raised = False
    except ValueError:
        raised = True
    checks.append(("Y >= 1: no fixed plan exists (edge case raises ValueError)", raised))

    # --- sensitivity to cycle length
    def delay_at(Cc):
        spare = Cc - L.LOST
        return L.mean_over_seeds(q, lambda: L.FixedPlan((spare * 2 / 3, spare / 3)), seeds=range(1, 7))[0]
    d35, d100 = delay_at(35.0), delay_at(100.0)
    checks.append((f"a cycle of 100 s gives much more delay than Webster's 35 s ({d100:.1f} against {d35:.1f} s)", d100 > 1.5 * d35))
    d15 = L.mean_over_seeds(q, lambda: L.FixedPlan((6.0, 3.0)), seeds=range(1, 4))[0]
    checks.append((f"a 15 s cycle (x = 1 on both roads, the shortest possible) is worse than 35 s ({d15:.1f} against {d35:.1f} s)",
                   d15 > 1.05 * d35))

    # --- fixed against actuated
    d_act = L.mean_over_seeds(q, L.Actuated, seeds=range(1, 7))[0]
    checks.append((f"actuated rule lowers the delay of the Webster fixed plan by at least 20 percent ({d_act:.1f} against {d35:.1f} s)",
                   d_act < 0.8 * d35))

    # --- actuated rule: first green of phase 0 under two scripted queue situations
    def first_green(queues):
        ctl = L.Actuated(min_green=8.0, max_green=45.0)
        t, length = 0.0, 0.0
        while t < 200.0:
            ph, green = ctl(t, queues)
            if not (ph == 0 and green):
                return length
            length += 0.5
            t += 0.5
        return length
    checks.append(("actuated: empty queue on green and a car waiting on the other road -> green lasts the minimum, 8 s",
                   abs(first_green([0, 4]) - 8.0) <= 0.5))
    checks.append(("actuated: both roads busy -> green stops at the maximum, 45 s", abs(first_green([5, 5]) - 45.0) <= 0.5))
    checks.append(("actuated: nobody waiting on the other road -> green is kept", first_green([0, 0]) >= 199.0))

    # --- failure case: oversaturation
    r = L.simulate([0.35, 0.20], plan60, T=7200.0)
    half = len(r["qlen"]) // 2
    w1, w2 = r["qlen"][half - 1].sum(), r["qlen"][-1].sum()
    checks.append((f"oversaturation: queue grows without bound ({w1:.0f} cars after 1 h, {w2:.0f} after 2 h)", w2 > 1.8 * w1 and w1 > 100))
    ra = L.simulate([0.30, 0.10], plan60, poisson=False, T=3600.0)
    rate = ra["qlen"][-1, 0] / 3600.0
    checks.append((f"overloaded approach grows near q - s g / C = 0.075 cars/s (got {rate:.3f})", abs(rate - 0.075) < 0.012))

    # --- four-approach mini-project numbers
    q4 = [360 / 900, 270 / 900, 315 / 900, 225 / 900]
    plan4 = L.webster_plan(q4, s=1.0)
    checks.append(("mini-project: Webster cycle for 360/270/315/225 cars per 15 min is 46.7 s",
                   abs(plan4.cycle - 46.6667) < 1e-3 and abs(plan4.g[0] - 23.2381) < 1e-3 and abs(plan4.g[1] - 17.4286) < 1e-3))
    dw = L.mean_over_seeds(q4, lambda: plan4, seeds=range(1, 7), s=1.0)[0]
    de = L.mean_over_seeds(q4, lambda: L.FixedPlan((27.0, 27.0)), seeds=range(1, 7), s=1.0)[0]
    checks.append((f"mini-project: Webster timing cuts delay by at least 15 percent against 27/27 ({dw:.1f} against {de:.1f} s)", dw < 0.85 * de))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
